In [1]:
# ============================================================
# CELL 0 — Dependency Installation
# ============================================================
!apt-get update -qq && apt-get install -y -qq libicu-dev build-essential
!pip install -q fasttext polyglot pyicu morfessor

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 14.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
import re
import os
import urllib.request
import fasttext
import logging

try:
    from polyglot.detect import Detector as PolyglotDetector
    HAS_POLYGLOT = True
except ImportError:
    HAS_POLYGLOT = False

logging.getLogger('polyglot.detect.base').setLevel(logging.ERROR)

_orig_array = np.array

def _patched_array(obj, *args, **kwargs):
    if kwargs.get('copy') is False:
        kwargs['copy'] = None
    return _orig_array(obj, *args, **kwargs)

np.array = _patched_array

In [3]:
# ============================================================
# CELL 2 — Load models
# ============================================================
FT_MODEL_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"
FT_MODEL_PATH = "lid.176.ftz"
if not os.path.exists(FT_MODEL_PATH):
    urllib.request.urlretrieve(FT_MODEL_URL, FT_MODEL_PATH)
ft_model = fasttext.load_model(FT_MODEL_PATH)

In [4]:
# ============================================================
# CELL 3 — Load data
# ============================================================
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

df = pd.read_parquet('sessions_lang_transcript.parquet')
df.head(1)

gamesession_id  user_id  game_name model_type           created_at  \
0       140597768   830382  Minecraft      gen10  2026-08-04 13:05:17   

  lang_detected  lang_probability  \
0            en             0.999   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [5]:
# ============================================================
# CELL 4 — Word cleaning helper
# ============================================================
def clean_word(w):
    w = w.strip()
    w = re.sub(r"^[^\w']+|[^\w']+$", "", w, flags=re.UNICODE)
    return w

In [6]:
# ============================================================
# CELL 5 — FastText predictor
# ============================================================
def predict_fasttext(word):
    if word == '':
        return None
    labels, probs = ft_model.predict(word, k=1)
    lang = labels[0].replace('__label__', '')
    return lang, float(probs[0])

In [7]:
# ============================================================
# CELL 6 — Polyglot predictor
# ============================================================
def predict_polyglot(word):
    if not HAS_POLYGLOT or word == '':
        return None
    try:
        detector = PolyglotDetector(word, quiet=True)
        top = detector.language
        return top.code, float(top.confidence) / 100
    except Exception:
        return None

In [8]:
# ============================================================
# CELL 7 — Select target session
# ============================================================
TARGET_ID = 140597768

df_target = df[df['gamesession_id'] == TARGET_ID].reset_index(drop=True)
row = df_target.iloc[0]
segments = row['transcript_segments']

print(f"Jumlah segment: {len(segments)}")

Jumlah segment: 1058


In [9]:
# ============================================================
# CELL 8 — Word-level table builder
# ============================================================
def build_word_table(segments, predict_fn, session_id):
    rows = []
    for seg_idx, seg in enumerate(segments):
        for w in seg['words']:
            cw = clean_word(w['text'])
            if cw == '':
                continue
            res = predict_fn(cw)
            lang, score = res if res is not None else (None, None)
            rows.append({
                'id': session_id,
                'segment_number': seg_idx,
                'timestamp': w['timestamp'].tolist() if hasattr(w['timestamp'], 'tolist') else w['timestamp'],
                'word': cw,
                'language_detected': lang,
                'confidence': f"{score * 100:.2f}%" if score is not None else None
            })
    return pd.DataFrame(rows)

In [10]:
# ============================================================
# CELL 9 — Segment-level table builder
# ============================================================

def build_segment_table(segments, predict_fn, session_id):
    rows = []

    for seg_idx, seg in enumerate(segments):

        segment_text = seg['text'].strip()

        if segment_text == '':
            rows.append({
                'id': session_id,
                'segment_number': seg_idx,
                'segment_text': segment_text,
                'language_detected': None,
                'confidence': None
            })
            continue

        res = predict_fn(segment_text)

        if res is None:
            lang, score = None, None
        else:
            lang, score = res

        rows.append({
            'id': session_id,
            'segment_number': seg_idx,
            'segment_text': segment_text,
            'language_detected': lang,
            'confidence': f"{score * 100:.2f}%" if score is not None else None
        })

    return pd.DataFrame(rows)

In [11]:
# ============================================================
# CELL 10 — Session-level table builder
# ============================================================

def build_session_table(segment_df, session_id):

    valid = segment_df.dropna(subset=['language_detected'])

    if len(valid) == 0:
        return pd.DataFrame([{
            'id': session_id,
            'language_detected': None
        }])

    lang_counts = valid['language_detected'].value_counts()

    total_segments = len(valid)

    lang_distribution = (
        lang_counts / total_segments
    )

    lang_str = " | ".join([
        f"{lang} {pct * 100:.2f}%"
        for lang, pct in lang_distribution.items()
    ])

    return pd.DataFrame([{
        'id': session_id,
        'language_detected': lang_str
    }])

In [12]:
# ============================================================
# CELL 11 — Model registry (Polyglot + FastText only)
# ============================================================
MODEL_REGISTRY = {
    'fasttext': predict_fasttext,
}

if HAS_POLYGLOT:
    MODEL_REGISTRY['polyglot'] = predict_polyglot

print("Model aktif:", list(MODEL_REGISTRY.keys()))

Model aktif: ['fasttext']


In [13]:
# ============================================================
# CELL 12 — Build & display word tables
# ============================================================
word_tables = {}

for model_name, predict_fn in MODEL_REGISTRY.items():
    print(f"Memproses word table: {model_name}")
    word_tables[model_name] = build_word_table(segments, predict_fn, TARGET_ID)

for model_name, table in word_tables.items():
    print(f"=== LEVEL WORD - {model_name.upper()} ===")
    display(table)

Memproses word table: fasttext
=== LEVEL WORD - FASTTEXT ===


,id,segment_number,timestamp,word,language_detected,confidence
0,140597768,0,"[0.0, 0.46]",Fire,en,12.45%
1,140597768,0,"[0.46, 0.74]",is,en,99.69%
2,140597768,0,"[0.74, 1.6]",traffic,en,52.17%
3,140597768,0,"[5.34, 7.12]",know,en,88.68%
4,140597768,0,"[7.12, 7.26]",what,en,12.45%
...,...,...,...,...,...,...
13009,140597768,1057,"[9140.64, 9140.74]",a,en,12.45%
13010,140597768,1057,"[9140.74, 9140.86]",good,en,12.45%
13011,140597768,1057,"[9140.86, 9141.3]",night,en,17.86%
13012,140597768,1057,"[9143.2, 9143.5]",See,en,12.45%


In [14]:
# ============================================================
# CELL 13 — Build & display segment tables
# ============================================================
segment_tables = {}

for model_name, predict_fn in MODEL_REGISTRY.items():
    print(f"Memproses segment table: {model_name}")
    segment_tables[model_name] = build_segment_table(segments, predict_fn, TARGET_ID)

for model_name, table in segment_tables.items():
    print(f"=== LEVEL SEGMENT - {model_name.upper()} ===")
    display(table)

Memproses segment table: fasttext
=== LEVEL SEGMENT - FASTTEXT ===


,id,segment_number,segment_text,language_detected,confidence
0,140597768,0,Fire is traffic. know what mean?,en,98.70%
1,140597768,1,Yeah. I know. right. We're going Alandro.,en,96.71%
2,140597768,2,"Welcome back, Welcome back, little knee guards.",en,89.16%
3,140597768,3,Oh my God. I don't know what compelled me to say that I'm sorry.,en,98.16%
4,140597768,4,"right, I'm good.",en,93.30%
...,...,...,...,...,...
1053,140597768,1053,"it's not coming to me, got no wisdom today, I",en,93.89%
1054,140597768,1054,now it's cooking yeah it's burning it's it's a little bit a little bit it's burnt just because it's a fem boy doesn't mean it's not still a dude you're,en,99.51%
1055,140597768,1055,"still gay where is the what's there go and then that guys just place inside in the background And if I go outside one of those signs are UFM boy, I'll quit this game.",en,97.76%
1056,140597768,1056,"All right. Bye, everybody.",en,92.67%


In [15]:
# ============================================================
# CELL 14 — Build & display session tables
# ============================================================
session_tables = {}

for model_name, table in segment_tables.items():
    session_tables[model_name] = build_session_table(table, TARGET_ID)

for model_name, table in session_tables.items():
    print(f"=== LEVEL SESSION - {model_name.upper()} ===")
    display(table)

=== LEVEL SESSION - FASTTEXT ===


,id,language_detected
0,140597768,en 99.43% | pl 0.19% | eu 0.09% | eo 0.09% | fr 0.09% | it 0.09%


In [16]:
# ============================================================
# CELL 15 — Find multi-language segment agreed by all models
# ============================================================

multi_lang_segment_sets = {}

for model_name, table in word_tables.items():
    valid = table.dropna(subset=['language_detected'])

    language_count_per_segment = (
        valid
        .groupby('segment_number')['language_detected']
        .nunique()
    )

    multi_lang_segments = set(
        language_count_per_segment[
            language_count_per_segment > 1
        ].index
    )

    multi_lang_segment_sets[model_name] = multi_lang_segments

    print(
        f"{model_name.upper()}: "
        f"{len(multi_lang_segments)} segment multi-bahasa"
    )

if len(multi_lang_segment_sets) > 0:
    common_segment_numbers = set.intersection(
        *multi_lang_segment_sets.values()
    )
else:
    common_segment_numbers = set()

common_segment_numbers = sorted(common_segment_numbers)

print(
    f"\nJumlah segmen multi-bahasa yang disepakati semua model: "
    f"{len(common_segment_numbers)}"
)

if len(common_segment_numbers) > 0:
    sample_segment_number = (
        pd.Series(common_segment_numbers)
        .sample(1, random_state=42)
        .iloc[0]
    )

    selection_reason = "multi-language menurut semua model"

else:
    all_multi_segments = set()

    for segment_set in multi_lang_segment_sets.values():
        all_multi_segments.update(segment_set)

    all_multi_segments = sorted(all_multi_segments)

    if len(all_multi_segments) > 0:

        sample_segment_number = (
            pd.Series(all_multi_segments)
            .sample(1, random_state=42)
            .iloc[0]
        )

        selection_reason = "multi-language menurut minimal satu model"

    else:
        available_segments = sorted(
            segment_tables[
                next(iter(segment_tables))
            ]['segment_number']
            .dropna()
            .unique()
        )

        if len(available_segments) == 0:
            raise ValueError("Tidak ada segment yang tersedia.")

        sample_segment_number = (
            pd.Series(available_segments)
            .sample(1, random_state=42)
            .iloc[0]
        )

        selection_reason = "fallback: random segment"


print(f"Segment number terpilih: {sample_segment_number}")
print(f"Alasan pemilihan: {selection_reason}")

FASTTEXT: 449 segment multi-bahasa

Jumlah segmen multi-bahasa yang disepakati semua model: 449
Segment number terpilih: 658
Alasan pemilihan: multi-language menurut semua model


In [17]:
# ============================================================
# CELL 16 — Show sample segment across models
# ============================================================
for model_name in MODEL_REGISTRY.keys():
    sample_segment = segment_tables[model_name][
        segment_tables[model_name]['segment_number'] == sample_segment_number
    ].reset_index(drop=True)

    sample_words = word_tables[model_name][
        word_tables[model_name]['segment_number'] == sample_segment_number
    ].reset_index(drop=True)

    print(f"=== SEGMENT LEVEL - {model_name.upper()} ===")
    display(sample_segment)

    print(f"=== WORD LEVEL - {model_name.upper()} ===")
    display(sample_words)

=== SEGMENT LEVEL - FASTTEXT ===


,id,segment_number,segment_text,language_detected,confidence
0,140597768,658,All of the add-ons. There's like seven mods in this. One of them is called Horror Craft.,en,98.83%


=== WORD LEVEL - FASTTEXT ===


,id,segment_number,timestamp,word,language_detected,confidence
0,140597768,658,"[5471.61, 5471.95]",All,en,12.45%
1,140597768,658,"[5471.95, 5472.13]",of,en,12.45%
2,140597768,658,"[5472.13, 5472.21]",the,en,72.65%
3,140597768,658,"[5472.21, 5472.37]",add,ru,64.67%
4,140597768,658,"[5472.37, 5472.47]",ons,fr,53.89%
5,140597768,658,"[5472.57, 5472.71]",There's,en,87.72%
6,140597768,658,"[5472.71, 5472.79]",like,en,94.25%
7,140597768,658,"[5472.79, 5473.07]",seven,en,81.53%
8,140597768,658,"[5473.07, 5473.37]",mods,en,12.45%
9,140597768,658,"[5473.37, 5473.53]",in,en,93.27%


In [18]:
# ============================================================
# CELL 17 — Custom text input
# ============================================================
custom_text = input("Masukkan kalimat yang ingin diuji: ")

Masukkan kalimat yang ingin diuji: I want to eat sate ayam in jakarta with temanku after pulang kerja


In [19]:
# ============================================================
# CELL 18 — Custom text table builders
# ============================================================

def build_segment_table_from_text(
    text,
    predict_fn,
    session_id,
    segment_number
):
    segment_text = text.strip()

    if segment_text == '':
        return pd.DataFrame([{
            'id': session_id,
            'segment_number': segment_number,
            'segment_text': segment_text,
            'language_detected': None,
            'confidence': None
        }])

    res = predict_fn(segment_text)

    if res is None:
        lang, score = None, None
    else:
        lang, score = res

    return pd.DataFrame([{
        'id': session_id,
        'segment_number': segment_number,
        'segment_text': segment_text,
        'language_detected': lang,
        'confidence': f"{score * 100:.2f}%" if score is not None else None
    }])


def build_word_table_from_text(
    text,
    predict_fn,
    session_id,
    segment_number
):
    words = text.split()
    rows = []

    for w in words:
        cw = clean_word(w)

        if cw == '':
            continue

        res = predict_fn(cw)
        lang, score = res if res is not None else (None, None)

        rows.append({
            'id': session_id,
            'segment_number': segment_number,
            'word': cw,
            'language_detected': lang,
            'confidence': f"{score * 100:.2f}%" if score is not None else None
        })

    return pd.DataFrame(rows)

In [20]:
# ============================================================
# CELL 19 — Run custom text through Polyglot + FastText
# ============================================================
CUSTOM_SESSION_ID = "custom_input"
CUSTOM_SEGMENT_NUMBER = 0

for model_name, predict_fn in MODEL_REGISTRY.items():
    custom_segment_table = build_segment_table_from_text(custom_text, predict_fn, CUSTOM_SESSION_ID, CUSTOM_SEGMENT_NUMBER)
    custom_word_table = build_word_table_from_text(custom_text, predict_fn, CUSTOM_SESSION_ID, CUSTOM_SEGMENT_NUMBER)

    print(f"=== SEGMENT LEVEL - {model_name.upper()} ===")
    display(custom_segment_table)

    print(f"=== WORD LEVEL - {model_name.upper()} ===")
    display(custom_word_table)

=== SEGMENT LEVEL - FASTTEXT ===


,id,segment_number,segment_text,language_detected,confidence
0,custom_input,0,I want to eat sate ayam in jakarta with temanku after pulang kerja,en,85.52%


=== WORD LEVEL - FASTTEXT ===


,id,segment_number,word,language_detected,confidence
0,custom_input,0,I,en,12.45%
1,custom_input,0,want,en,99.46%
2,custom_input,0,to,en,12.45%
3,custom_input,0,eat,en,99.49%
4,custom_input,0,sate,en,28.01%
5,custom_input,0,ayam,fr,13.71%
6,custom_input,0,in,en,93.27%
7,custom_input,0,jakarta,id,41.40%
8,custom_input,0,with,en,97.83%
9,custom_input,0,temanku,eo,26.33%


In [21]:
# ============================================================
# CELL 20 — Sliding window 4-word grouping builder
# ============================================================

def build_group4_table_from_text(
    text,
    predict_fn,
    session_id,
    segment_number
):
    words = text.split()
    rows = []

    n = len(words)
    window_size = 4

    if n == 0:
        return pd.DataFrame(rows)

    if n < window_size:
        window_size = n

    for i in range(0, n - window_size + 1):

        window_words = words[i:i + window_size]

        group_text = " ".join(window_words).strip()

        if group_text == '':
            continue

        res = predict_fn(group_text)

        if res is None:
            lang, score = None, None
        else:
            lang, score = res

        rows.append({
            'id': session_id,
            'segment_number': segment_number,
            'group_index': i + 1,
            'group_text': group_text,
            'language_detected': lang,
            'confidence': f"{score * 100:.2f}%" if score is not None else None
        })

    return pd.DataFrame(rows)

In [22]:
# ============================================================
# CELL 21 — Run custom text through Polyglot + FastText (segment + 4-word grouping)
# ============================================================
CUSTOM_SESSION_ID = "custom_input"
CUSTOM_SEGMENT_NUMBER = 0

for model_name, predict_fn in MODEL_REGISTRY.items():
    custom_segment_table = build_segment_table_from_text(custom_text, predict_fn, CUSTOM_SESSION_ID, CUSTOM_SEGMENT_NUMBER)
    custom_group4_table = build_group4_table_from_text(custom_text, predict_fn, CUSTOM_SESSION_ID, CUSTOM_SEGMENT_NUMBER)

    print(f"=== SEGMENT LEVEL - {model_name.upper()} ===")
    display(custom_segment_table)

    print(f"=== GROUP 4-WORD LEVEL - {model_name.upper()} ===")
    display(custom_group4_table)

=== SEGMENT LEVEL - FASTTEXT ===


,id,segment_number,segment_text,language_detected,confidence
0,custom_input,0,I want to eat sate ayam in jakarta with temanku after pulang kerja,en,85.52%


=== GROUP 4-WORD LEVEL - FASTTEXT ===


,id,segment_number,group_index,group_text,language_detected,confidence
0,custom_input,0,1,I want to eat,en,99.90%
1,custom_input,0,2,want to eat sate,en,99.66%
2,custom_input,0,3,to eat sate ayam,en,91.70%
3,custom_input,0,4,eat sate ayam in,en,96.28%
4,custom_input,0,5,sate ayam in jakarta,en,47.03%
5,custom_input,0,6,ayam in jakarta with,en,77.42%
6,custom_input,0,7,in jakarta with temanku,en,67.33%
7,custom_input,0,8,jakarta with temanku after,en,77.16%
8,custom_input,0,9,with temanku after pulang,en,80.68%
9,custom_input,0,10,temanku after pulang kerja,en,56.27%


In [23]:
# ============================================================
# CELL 22 — Prediksi bahasa per SEGMENT (utuh, tanpa split kalimat)
# ============================================================
def build_persegment_table_from_transcript(segments, predict_fn, session_id):
    rows = []
    for seg_idx, seg in enumerate(segments):
        seg_text = seg['text']
        words = seg_text.split()

        lang_count = {}
        lang_conf_sum = {}

        for w in words:
            cw = clean_word(w)
            if cw == '':
                continue
            res = predict_fn(cw)
            if res is None:
                continue
            lang, score = res
            lang_count[lang] = lang_count.get(lang, 0) + 1
            lang_conf_sum[lang] = lang_conf_sum.get(lang, 0.0) + score

        total_words = sum(lang_count.values())
        if total_words == 0:
            rows.append({
                'id': session_id,
                'segment_number': seg_idx,
                'segment_text': seg_text,
                'timestamp': seg['timestamp'],
                'language_detected': None,
                'confidence': None
            })
            continue

        sorted_langs = sorted(lang_count.items(), key=lambda x: x[1], reverse=True)
        lang_str_parts = []
        conf_str_parts = []
        for lang, count in sorted_langs:
            proportion = count / total_words
            avg_conf = lang_conf_sum[lang] / count
            lang_str_parts.append(f"{lang} {proportion*100:.2f}%")
            conf_str_parts.append(f"{lang} {avg_conf*100:.2f}%")

        rows.append({
            'id': session_id,
            'segment_number': seg_idx,
            'segment_text': seg_text,
            'timestamp': seg['timestamp'],
            'language_detected': " | ".join(lang_str_parts),
            'confidence': " | ".join(conf_str_parts)
        })
    return pd.DataFrame(rows)

In [24]:
# ============================================================
# CELL 23 — Run per-segment (utuh) untuk Polyglot + FastText
# ============================================================
persegment_tables = {}

for model_name, predict_fn in MODEL_REGISTRY.items():
    print(f"Memproses per-segment table: {model_name}")
    persegment_tables[model_name] = build_persegment_table_from_transcript(segments, predict_fn, TARGET_ID)

for model_name, table in persegment_tables.items():
    print(f"=== LEVEL PER-SEGMENT (UTUH) - {model_name.upper()} ===")
    display(table)

Memproses per-segment table: fasttext
=== LEVEL PER-SEGMENT (UTUH) - FASTTEXT ===


,id,segment_number,segment_text,timestamp,language_detected,confidence
0,140597768,0,Fire is traffic. know what mean?,"[0.0, 7.42]",en 100.00%,en 53.91%
1,140597768,1,Yeah. I know. right. We're going Alandro.,"[8.54, 11.84]",en 100.00%,en 49.79%
2,140597768,2,"Welcome back, Welcome back, little knee guards.","[12.42, 16.76]",en 85.71% | it 14.29%,en 26.16% | it 37.15%
3,140597768,3,Oh my God. I don't know what compelled me to say that I'm sorry.,"[17.1, 22.56]",en 100.00%,en 47.53%
4,140597768,4,"right, I'm good.","[27.38, 30.79]",en 100.00%,en 45.18%
...,...,...,...,...,...,...
1053,140597768,1053,"it's not coming to me, got no wisdom today, I","[9087.58, 9090.78]",en 90.00% | nl 10.00%,en 44.54% | nl 70.33%
1054,140597768,1054,now it's cooking yeah it's burning it's it's a little bit a little bit it's burnt just because it's a fem boy doesn't mean it's not still a dude you're,"[9101.14, 9119.2]",en 100.00%,en 62.09%
1055,140597768,1055,"still gay where is the what's there go and then that guys just place inside in the background And if I go outside one of those signs are UFM boy, I'll quit this game.","[9121.6, 9135.36]",en 97.06% | tt 2.94%,en 56.42% | tt 3.02%
1056,140597768,1056,"All right. Bye, everybody.","[9136.02, 9139.74]",en 100.00%,en 27.00%
